##### CONEXION

In [91]:
import importlib.util
import os
import pandas as pd
from datetime import datetime
import numpy as np
pd.set_option("display.precision", 10)

UTILS_PATH = r"C:\Users\User\OneDrive\Proyecto\PYTHON\Proyecto HPH\CROSS X\utils.py"

if not os.path.isfile(UTILS_PATH):
    raise FileNotFoundError(UTILS_PATH)

spec = importlib.util.spec_from_file_location("util", UTILS_PATH)
utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(utils)

##### INSTANCIA DE LOS DATAFRAMES

In [92]:
df_ds = utils.dataframe_DS()
dataframe_DS_Virgen = utils.dataframe_DS_Virgen()
df_cli=utils.dataframe_Cliente()
df_homo=utils.homologa()

C:\Users\User\OneDrive\Proyecto\PYTHON\Proyecto HPH\CROSS X\utils.py:304: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_v512["ADU-PAT-PED"] = (df_v512["SeccionAduanera"].astype(str)+ "-"+ df_v512["Patente"].astype(str)+ "-"+ df_v512["Pedimento"].astype(str))
C:\Users\User\OneDrive\Proyecto\PYTHON\Proyecto HPH\CROSS X\utils.py:305: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_v512["ADU-PAT-PED ANTERIOR"] = (df_v512["SeccionAduaneraDespOrig"].astype(str) + "-" + df_v512["PatenteAduanalOrig"].astype(str

UnidadMedida vacíos: 0
MercanciaDescargada vacíos: 0


In [93]:

columnas_group_cli = [
    "ADU-PAT-PED-HTS"
]

detalle_umc_cli = (
    df_cli.groupby(
        columnas_group_cli + ["UM NORMALIZADA"],
        as_index=False
    )["CANT_UMC_NEW"]
    .sum()
)

detalle_umc_cli = (
    detalle_umc_cli.groupby(columnas_group_cli, as_index=False)
    .agg({
        "CANT_UMC_NEW": lambda x: " & ".join(x.astype(str))
    })
    .rename(columns={"CANT_UMC_NEW": "CANT_UMC_NEW_DETALLE"})
)

df_cli_group = (
    df_cli.groupby(columnas_group_cli, as_index=False)
    .agg({
        "VALOR EN DOLARES": "sum",
        "CANT_UMC_NEW": "sum",
        "UM NORMALIZADA": lambda x: " & ".join(dict.fromkeys(x.astype(str)))
    })
)

df_cli_group = df_cli_group.merge(
    detalle_umc_cli,
    on=columnas_group_cli,
    how="left"
)

In [94]:
df_cli_group

# display(
#     df_cli_group[
#         df_cli_group["ADU-PAT-PED-HTS"] == "240-1660-4008796-98020019"
        
#     ]
# )



,ADU-PAT-PED-HTS,VALOR EN DOLARES,CANT_UMC_NEW,UM NORMALIZADA,CANT_UMC_NEW_DETALLE
0,160-1634-3000040-84148099,53537.91,133.0,PC,133.0
1,160-1634-3000346-84148099,19119.39,51.0,PC,51.0
2,160-1634-3000605-84148099,37489.00,100.0,PC,100.0
3,160-1634-3001257-84148099,34114.99,91.0,PC,91.0
4,160-1634-3002852-84148099,23618.07,63.0,PC,63.0
...,...,...,...,...,...
31577,VME082024BTE0007-99999999,0.00,5712.0,PC,5712.0
31578,VME092024BTE0008-99999999,0.00,5400.0,PC,5400.0
31579,VME102024BTE0009-99999999,0.00,7824.0,PC,7824.0
31580,VME112024BTE0010-99999999,0.00,8208.0,PC,8208.0


##### DAR EL FORMATO INICIAL PARA CROSS X HTS

In [95]:

df_ds["DS PAT-PED"] = (
    df_ds["ADU-PAT-PED"]
    .str.split("-", n=1)
    .str[1]
)

df_ds["DS PAT-PED-HTS"] = (
    df_ds["ADU-PAT-PED-HTS"]
    .str.split("-", n=1)
    .str[1]
)

df_ds["DS PAT-PED-HTS"] = (
    df_ds["ADU-PAT-PED-HTS"]
    .str.split("-", n=1)
    .str[1]
)


# df_ds["DS ADU-PAT-PED-HTS"] = (
#     df_ds["DS ADU-PAT-PED"]
#     + "-"
#     + df_ds["DS PAT-PED-HTS"].str.rsplit("-", n=1).str[-1]
# )

##### CREAR DATOS BASICOS PARA CROSS X HTS

In [96]:

# total_val = df_ds["ADU-PAT-PED"].nunique()
# print(f"Registros Totales Unicos: {total_val}")

columnas_group = [
    "DS PAT-PED",
    "DS PAT-PED-HTS",
    "ADU-PAT-PED",
    "ADU-PAT-PED-HTS",
    "CLAVE PED",
    "FECHA PAGO 551"
]

detalle_umc = (
    df_ds.groupby(
        columnas_group + ["UM NORMALIZADA"],
        as_index=False
    )["CANT_UMC_NEW"]
    .sum()
)

detalle_umc = (
    detalle_umc.groupby(columnas_group, as_index=False)
    .agg({
        "CANT_UMC_NEW": lambda x: " & ".join(x.astype(str))
    })
    .rename(columns={"CANT_UMC_NEW": "CANT_UMC_NEW_DETALLE"})
)

df_ds_group = (
    df_ds.groupby(columnas_group, as_index=False)
    .agg({
        "VAL USD": "sum",
        "CANT_UMC_NEW": "sum",
        "UM NORMALIZADA": lambda x: " & ".join(dict.fromkeys(x.astype(str)))
    })
)

df_ds_group = df_ds_group.merge(
    detalle_umc,
    on=columnas_group,
    how="left"
)

##### EXTRAER INFORMACION DEL CROSS X PED

In [97]:
ruta= r"C:/Users/User/OneDrive/Proyecto/PYTHON/Proyecto HPH/CROSS X/outputs/"
df_ped = utils.leer_csv(ruta + "CROSS X PED.csv")

In [98]:
df_ped = df_ped[
    [
        "DS ADU-PAT-PED",
        "CLAVES IMMEX",
        "FECHA PAGO",
        "FECHA ENTRADA O PRESENTACIÓN",
        "ID AF?",
        "TIPO Y CLAVE",
        "SI EL PEDIMENTO DE LA COLUMNA 'T' ES R1, A QUIEN RECTIFICÓ?",
        "ESTATUS DEL PEDIMENTO A QUIEN RECTIFICO DE LA COLUMNA 'F'?",
        "SI EL PEDIMENTO DE LA COLUMNA 'T' TUVO R1, CUÁL FUE EL NÚMERO DE R1?",
        "ESTATUS DEL PEDIMENTO R1 DE LA COLUMNA H",
        "ESTIMACIÓN APROX DE IGI  6 FP 0",
        "ESTIMACIÓN APROX DE IGI 6 FP 5",
        "ESTIMACIÓN APROX DE IGI 6 FP 6",
        "SI EL PEDIMENTO DE LA COLUMNA 'T' ES DESCARGA ESPECIFICA, A QUIÉN DESCARGO?",
        "SI EL PEDIMENTO DE LA COLUMNA 'Q' TUVO DESCARGAR ESPECIFICA, EN QUE PEDIMENTO SE DESCARGO?",
        "OBSERVACIONES GLOBALES YA CON R1 (ETIQUETA ANTERIOR)",
    ]
].rename(
    columns={
        "SI EL PEDIMENTO DE LA COLUMNA 'T' ES R1, A QUIEN RECTIFICÓ?": "SI EL PEDIMENTO DE LA COLUMNA 'Q' ES R1, A QUIEN RECTIFICÓ?",
        "SI EL PEDIMENTO DE LA COLUMNA 'T' TUVO R1, CUÁL FUE EL NÚMERO DE R1?": "SI EL PEDIMENTO DE LA COLUMNA 'Q' TUVO R1, CUÁL FUE EL NÚMERO DE R1?",
        "SI EL PEDIMENTO DE LA COLUMNA 'T' ES DESCARGA ESPECIFICA, A QUIÉN DESCARGO?":"SI EL PEDIMENTO DE LA COLUMNA 'Q' ES DESCARGA ESPECIFICA, A QUIÉN DESCARGO?"
    }
)


df_ped2 = df_ped[
    [
        
        "DS ADU-PAT-PED",
        "OBSERVACIONES GLOBALES YA CON R1 (ETIQUETA ANTERIOR)",
    ]
]


# for c in df_ped.columns:
#     print(c)
    
# df_ped

In [99]:
df_ds_group = df_ds_group.merge(
    df_ped,
    left_on="ADU-PAT-PED",
    right_on="DS ADU-PAT-PED",
    how="left"
)

df_ds_group.drop(columns=["DS ADU-PAT-PED"], inplace=True)

In [100]:
df_ped

,DS ADU-PAT-PED,CLAVES IMMEX,FECHA PAGO,FECHA ENTRADA O PRESENTACIÓN,ID AF?,TIPO Y CLAVE,"SI EL PEDIMENTO DE LA COLUMNA 'Q' ES R1, A QUIEN RECTIFICÓ?",ESTATUS DEL PEDIMENTO A QUIEN RECTIFICO DE LA COLUMNA 'F'?,"SI EL PEDIMENTO DE LA COLUMNA 'Q' TUVO R1, CUÁL FUE EL NÚMERO DE R1?",ESTATUS DEL PEDIMENTO R1 DE LA COLUMNA H,ESTIMACIÓN APROX DE IGI 6 FP 0,ESTIMACIÓN APROX DE IGI 6 FP 5,ESTIMACIÓN APROX DE IGI 6 FP 6,"SI EL PEDIMENTO DE LA COLUMNA 'Q' ES DESCARGA ESPECIFICA, A QUIÉN DESCARGO?","SI EL PEDIMENTO DE LA COLUMNA 'Q' TUVO DESCARGAR ESPECIFICA, EN QUE PEDIMENTO SE DESCARGO?",OBSERVACIONES GLOBALES YA CON R1 (ETIQUETA ANTERIOR)
0,160-1628-9003233,UNIVERSO IMMEX,2/04/2019,2/04/2019,NaN,2-RT-TUVO R1-160-1628-9003631,NO ES R1,NO SE REQUIERE,160-1628-9003631,UNIVERSO IMMEX-ES R1 Y TUVO A SU VEZ R1-PEDIME...,NaN,NaN,NaN,NO ES DESCARGA,NO DESCARGO,UNIVERSO IMMEX-NI EL ORIGINAL NI EL R1 ESTAN C...
1,160-1628-9003631,UNIVERSO IMMEX,10/04/2019,2/04/2019,NaN,2-RT-R1-160-1628-9003233-TUVO R1-160-1628-9003713,160-1628-9003233,UNIVERSO IMMEX-TUVO R1-PEDIMENTO FALTANTE EN BASE,160-1628-9003713,UNIVERSO IMMEX-ES R1-PEDIMENTO FALTANTE EN BASE,NaN,NaN,NaN,NO ES DESCARGA,NO DESCARGO,UNIVERSO IMMEX-NI EL ORIGINAL NI EL R1 ESTAN C...
2,160-1628-9003713,UNIVERSO IMMEX,12/04/2019,2/04/2019,NaN,2-RT-R1-160-1628-9003631,160-1628-9003631,UNIVERSO IMMEX-ES R1 Y TUVO A SU VEZ R1-PEDIME...,NO TUVO R1,NO SE REQUIERE,NaN,NaN,NaN,NO ES DESCARGA,NO DESCARGO,"NI EL R1 NI EL ORIGINAL ESTAN BIEN, R1 CON EST..."
3,521-1629-0000006,UNIVERSO IMMEX,21/01/2020,15/01/2020,NaN,2-RT,NO ES R1,NO SE REQUIERE,NO TUVO R1,NO SE REQUIERE,NaN,NaN,NaN,NO ES DESCARGA,NO DESCARGO,UNIVERSO IMMEX-PEDIMENTO FALTANTE EN BASE
4,521-1629-0000036,UNIVERSO IMMEX,4/03/2020,27/02/2020,NaN,2-RT,NO ES R1,NO SE REQUIERE,NO TUVO R1,NO SE REQUIERE,NaN,NaN,NaN,NO ES DESCARGA,NO DESCARGO,UNIVERSO IMMEX-PEDIMENTO FALTANTE EN BASE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44887,240-3989-9012334,UNIVERSO IMMEX,3/12/2019,27/11/2019,NaN,2-RT,NO ES R1,NO SE REQUIERE,NO TUVO R1,NO SE REQUIERE,NaN,NaN,NaN,NO ES DESCARGA,NO DESCARGO,UNIVERSO IMMEX-PEDIMENTO FALTANTE EN BASE
44888,240-3989-9012484,UNIVERSO IMMEX,10/12/2019,4/12/2019,NaN,2-RT,NO ES R1,NO SE REQUIERE,NO TUVO R1,NO SE REQUIERE,NaN,NaN,NaN,NO ES DESCARGA,NO DESCARGO,UNIVERSO IMMEX-PEDIMENTO FALTANTE EN BASE
44889,240-3989-9012724,NO IMMEX,17/12/2019,11/12/2019,NaN,2-A1,NO ES R1,NO SE REQUIERE,NO TUVO R1,NO SE REQUIERE,NaN,NaN,NaN,NO ES DESCARGA,NO DESCARGO,NO IMMEX-PEDIMENTO FALTANTE EN BASE
44890,240-3989-9012950,UNIVERSO IMMEX,26/12/2019,18/12/2019,NaN,2-RT,NO ES R1,NO SE REQUIERE,NO TUVO R1,NO SE REQUIERE,NaN,NaN,NaN,NO ES DESCARGA,NO DESCARGO,UNIVERSO IMMEX-PEDIMENTO FALTANTE EN BASE


##### CRUCE ENTRE DATAFRAME DS VS CLIENTE

In [101]:
df_ds_group = df_ds_group.merge(
    df_cli_group[
        ["ADU-PAT-PED-HTS", "VALOR EN DOLARES","UM NORMALIZADA","CANT_UMC_NEW_DETALLE"]
    ].rename(
        columns={
            "VALOR EN DOLARES": "BASE VALOR DOLARES",
            "CANT_UMC_NEW_DETALLE": "BASE CANTIDAD COMERCIAL",
            "UM NORMALIZADA": "BASE UNIDAD DE MEDIDA COMERCIAL"
        }
    ),
    left_on="ADU-PAT-PED-HTS",
    right_on="ADU-PAT-PED-HTS",
    how="left"
)





In [102]:

dif_real = df_ds_group["VAL USD"] - df_ds_group["BASE VALOR DOLARES"].fillna(0)

df_ds_group["DIFERENCIA REAL"] = np.where(
    dif_real.between(-1, 1),
    0,
    dif_real
)

df_ds_group["DIFERENCIA ABSOLUTA"] = df_ds_group["DIFERENCIA REAL"].abs()


df_ds_group["REVISIÓN DE DUPLICADOS"] = np.where(
    (df_ds_group["BASE VALOR DOLARES"].isna()) |
    (df_ds_group["VAL USD"] < 2),
    "",
    np.where(
        (
            (df_ds_group["VAL USD"] - df_ds_group["DIFERENCIA ABSOLUTA"] > -1)
            &
            (df_ds_group["VAL USD"] - df_ds_group["DIFERENCIA ABSOLUTA"] < 1)
            &
            (df_ds_group["BASE VALOR DOLARES"] > 3)
        ),
        "DUPLICADO",
        ""
    )
)

In [103]:
# pd.set_option("display.max_columns", None)
# # df_ds_group
# # df_ds_group

# # display(df_ds_group.columns.tolist())

# display(
#     df_ds_group[
#         df_ds_group["ADU-PAT-PED"] == "750-3694-3010589"
        
#     ]
# )



In [104]:

def ordenar(valor):
    if pd.isna(valor):
        return valor
    return " & ".join(sorted(str(valor).split(" & ")))

x = df_ds_group["CANT_UMC_NEW_DETALLE"].apply(ordenar)
y = df_ds_group["UM NORMALIZADA"].apply(ordenar)
z = df_ds_group["BASE CANTIDAD COMERCIAL"].apply(ordenar)
aa = df_ds_group["BASE UNIDAD DE MEDIDA COMERCIAL"].apply(ordenar)

cant = pd.to_numeric(df_ds_group["CANT_UMC_NEW_DETALLE"], errors="coerce")
base = pd.to_numeric(df_ds_group["BASE CANTIDAD COMERCIAL"], errors="coerce")

resultado = np.empty(len(df_ds_group), dtype=object)

# SI(Y(Y=AA;X=Z);0)
resultado[(y == aa) & (x == z)] = 0

# SI(Z="";X)
m = ~( (y == aa) & (x == z) ) & base.isna()
resultado[m] = df_ds_group.loc[m, "CANT_UMC_NEW_DETALLE"]

# SI(Y<>AA;"REVISAR UM")
m = ~( (y == aa) & (x == z) ) & ~base.isna() & (y != aa)
resultado[m] = "REVISAR UM"

# Diferencias numéricas
m = ~( (y == aa) & (x == z) ) & ~base.isna() & (y == aa)

dif = cant - base

resultado[m & dif.between(-1, 1)] = 0
resultado[m & ~dif.between(-1, 1)] = dif[m & ~dif.between(-1, 1)]

df_ds_group["DIFERENCIA REAL UND"] = resultado


df_ds_group["DIFERENCIA ABSOLUTA UND"] = pd.to_numeric(df_ds_group["DIFERENCIA REAL UND"], errors="coerce").abs().fillna(df_ds_group["DIFERENCIA REAL UND"])

x = pd.to_numeric(df_ds_group["CANT_UMC_NEW_DETALLE"], errors="coerce")
z = pd.to_numeric(df_ds_group["BASE CANTIDAD COMERCIAL"], errors="coerce")
ac = pd.to_numeric(df_ds_group["DIFERENCIA ABSOLUTA UND"], errors="coerce")

df_ds_group["REVISIÓN DE DUPLICADOS UND"] = np.where(
    x.isna() | z.isna() | (z == "") | (x <= 2),
    "",
    np.where(
        ((x - ac) > -1) &
        ((x - ac) < 1) &
        (z > 3),
        "DUPLICADO",
        ""
    )
)

In [105]:

s = pd.to_numeric(df_ds_group["VAL USD"], errors="coerce")
t = pd.to_numeric(df_ds_group["BASE VALOR DOLARES"], errors="coerce")
u = pd.to_numeric(df_ds_group["DIFERENCIA REAL"], errors="coerce")

df_ds_group["ESTATUS USD A NIVEL HTS AL 15.05.26"] = np.where(
    df_ds_group["REVISIÓN DE DUPLICADOS"] == "DUPLICADO",
    df_ds_group["CLAVES IMMEX"] + "-DUPLICADO",

    np.where(
        t.isna() | (t.astype(str).str.strip() == ""),
        df_ds_group["CLAVES IMMEX"] + "-FRACCION FALTANTE EN BASE",

        np.where(
            (u >= -1) & (u <= 1),
            df_ds_group["CLAVES IMMEX"] + "-100% CORRECTO EN VALOR HTS",

            np.where(
                (t == 0) & (s > 1),
                df_ds_group["CLAVES IMMEX"] + "-LE FALTA VALOR HTS EN BASE",

                np.where(
                    (((s / t) - 1) >= -0.01) &
                    (((s / t) - 1) <= 0.01),
                    df_ds_group["CLAVES IMMEX"] + "-99% CORRECTO EN VALOR HTS",

                    np.where(
                        t > s,
                        df_ds_group["CLAVES IMMEX"] + "-LE SOBRA VALOR HTS EN BASE",

                        np.where(
                            t < s,
                            df_ds_group["CLAVES IMMEX"] + "-LE FALTA VALOR HTS EN BASE",
                            ""
                        )
                    )
                )
            )
        )
    )
)

In [106]:
# x = pd.to_numeric(df_ds_group["CANT_UMC_NEW_DETALLE"], errors="coerce")
# z = pd.to_numeric(df_ds_group["BASE CANTIDAD COMERCIAL"], errors="coerce")
ab = pd.to_numeric(df_ds_group["DIFERENCIA REAL UND"], errors="coerce")


x = pd.to_numeric(df_ds_group["CANT_UMC_NEW_DETALLE"], errors="coerce").where(
    pd.to_numeric(df_ds_group["CANT_UMC_NEW_DETALLE"], errors="coerce").notna(),
    df_ds_group["CANT_UMC_NEW_DETALLE"]
)

z = pd.to_numeric(df_ds_group["BASE CANTIDAD COMERCIAL"], errors="coerce").where(
    pd.to_numeric(df_ds_group["BASE CANTIDAD COMERCIAL"], errors="coerce").notna(),
    df_ds_group["BASE CANTIDAD COMERCIAL"]
)


z_vacio = z.isna() | (df_ds_group["BASE CANTIDAD COMERCIAL"].astype(str).str.strip() == "")

df_ds_group["ESTATUS CANTIDAD COMERCIAL A NIVEL HTS AL 15.05.26"] = np.where(
    df_ds_group["DIFERENCIA REAL UND"] == "REVISAR UM",
    df_ds_group["CLAVES IMMEX"] + "-REVISAR UM",

    np.where(
        df_ds_group["REVISIÓN DE DUPLICADOS UND"] == "DUPLICADO",
        df_ds_group["CLAVES IMMEX"] + "-DUPLICADO",

        np.where(
            z_vacio,
            df_ds_group["CLAVES IMMEX"] + "-FRACCION FALTANTE EN BASE",

            np.where(
                (ab >= -1) & (ab <= 1),
                df_ds_group["CLAVES IMMEX"] + "-100% CORRECTO EN CANTIDAD HTS",

                np.where(
                    # x > 1,
                    pd.to_numeric(x, errors="coerce").fillna(0) > 1,
                    df_ds_group["CLAVES IMMEX"] + "-LE FALTA CANTIDAD HTS EN BASE",

                    # np.where(
                    #     (((x / z) - 1) >= -0.01) &
                    #     (((x / z) - 1) <= 0.01),
                    #     df_ds_group["CLAVES IMMEX"] + "-99% CORRECTO EN CANTIDAD HTS EN BASE",

                    np.where(
                        (
                            (
                                pd.to_numeric(x, errors="coerce").fillna(0) /
                                pd.to_numeric(z, errors="coerce").replace(0, np.nan)
                            ) - 1 >= -0.01
                        ) &
                        (
                            (
                                pd.to_numeric(x, errors="coerce").fillna(0) /
                                pd.to_numeric(z, errors="coerce").replace(0, np.nan)
                            ) - 1 <= 0.01
                        ),
                        df_ds_group["CLAVES IMMEX"] + "-99% CORRECTO EN CANTIDAD HTS EN BASE",



                        # np.where(
                        #     z > x,
                        #     df_ds_group["CLAVES IMMEX"] + "-LE SOBRA CANTIDAD HTS EN BASE",

                        #     np.where(
                        #         z < x,
                        #         df_ds_group["CLAVES IMMEX"] + "-LE FALTA CANTIDAD HTS EN BASE",
                        #         ""
                        
                        np.where(
                            pd.to_numeric(z, errors="coerce").fillna(0) >
                            pd.to_numeric(x, errors="coerce").fillna(0),

                            df_ds_group["CLAVES IMMEX"] + "-LE SOBRA CANTIDAD HTS EN BASE",

                            np.where(
                                pd.to_numeric(z, errors="coerce").fillna(0) <
                                pd.to_numeric(x, errors="coerce").fillna(0),

                                df_ds_group["CLAVES IMMEX"] + "-LE FALTA CANTIDAD HTS EN BASE",
                                ""                        

                        
                            )
                        )
                    )
                )
            )
        )
    )
)

In [107]:
# display(
#     df_ds_group[
#         df_ds_group["ADU-PAT-PED-HTS"] == "240-1660-4008796-98020019"
        
#     ]
# )

In [108]:

t_vacio = (
    df_ds_group["BASE VALOR DOLARES"].isna() |
    (df_ds_group["BASE VALOR DOLARES"].astype(str).str.strip() == "")
)

z_vacio = (
    df_ds_group["BASE CANTIDAD COMERCIAL"].isna() |
    (df_ds_group["BASE CANTIDAD COMERCIAL"].astype(str).str.strip() == "")
)

v = pd.to_numeric(df_ds_group["DIFERENCIA ABSOLUTA"], errors="coerce")
ab = pd.to_numeric(df_ds_group["DIFERENCIA ABSOLUTA UND"], errors="coerce")

df_ds_group["ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 1)"] = np.where(
    t_vacio | z_vacio,
    "FRACCION FALTANTE EN BASE",

    np.where(
        (v == 0) & (ab == 0),
        "FRACCION 100% CORRECTA EN DOLARES Y CANTIDAD",

        np.where(
            (v == 0) & (ab != 0),
            "FRACCION BIEN EN DOLARES Y MAL EN CANTIDAD",

            np.where(
                (v != 0) & (ab == 0),
                "FRACCION BIEN EN CANTIDAD Y MAL EN DOLARES",

                np.where(
                    (v != 0) & (ab != 0),
                    "FRACCION REQUIERE CORRECCION",
                    ""
                )
            )
        )
    )
)

In [109]:
estado_ok = "FRACCION 100% CORRECTA EN DOLARES Y CANTIDAD"

# Obtener los ADU-PAT-PED donde TODAS las fracciones tienen el estado correcto
pedimentos_ok = (
    df_ds_group
    .groupby("ADU-PAT-PED")["ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 1)"]
    .apply(lambda x: (x == estado_ok).all())
)

# Quedarse solo con los ADU-PAT-PED válidos
pedimentos_ok = pedimentos_ok[pedimentos_ok].index

# Crear el nuevo DataFrame
df_ped_aux = (
    df_ds_group[
        df_ds_group["ADU-PAT-PED"].isin(pedimentos_ok)
    ][
        [
            "ADU-PAT-PED",
            "ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 1)"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


df_ds_group = df_ds_group.merge(
    df_ped_aux.rename(columns={
        "ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 1)": "ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 1)_aux"
    }),
    on="ADU-PAT-PED",
    how="left"
)

# df_ds_group = df_ds_group.merge(
#     df_ped_aux,
#     on="ADU-PAT-PED",
#     how="left"
# )

df_ds_group["ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 2)"] = np.where(
    df_ds_group["ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 1)_aux"] == "FRACCION 100% CORRECTA EN DOLARES Y CANTIDAD",
    "PEDIMENTO CON HTS 100% CORRECTOS DOLARES Y CANTIDAD",
    ""
)

df_ds_group.drop(columns=["ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 1)_aux"], inplace=True)


In [110]:
df_ds_group["ESTATUS HTS A NIVEL PEDIMENTO"] = np.where(
    df_ds_group["ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 2)"].fillna("").str.strip() != "",
    df_ds_group["ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 2)"],
    df_ds_group["ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 1)"]
)


# df_ds_group["LLAVE PED-VALORUSD"] = (
#     df_ds_group["ADU-PAT-PED"].astype(str)
#     + "-"
#     + np.floor(pd.to_numeric(df_ds_group["VAL USD"], errors="coerce")).fillna(0).astype(int).astype(str)
# )

df_ds_group["LLAVE PED-VALORUSD"] = (
    df_ds_group["ADU-PAT-PED"].astype(str)
    + "-"
    + np.floor(
        pd.to_numeric(df_ds_group["VAL USD"], errors="coerce") + 1e-9
    ).fillna(0).astype(int).astype(str)
)

##### FORMATEANDO ANTES DE EXPORTAR

In [111]:
df_ds_group = (
    df_ds_group
    .rename(columns={
        "ADU-PAT-PED": "DS ADU-PAT-PED",
        "ADU-PAT-PED-HTS": "DS ADU-PAT-PED-HTS",
        "VAL USD":"DS VALOR DOLARES",
        "CANT_UMC_NEW_DETALLE":"DS CANTIDAD COMERCIAL",
        "UM NORMALIZADA":"DS UNIDAD DE MEDIDA COMERCIAL",
        "ESTATUS USD A NIVEL HTS AL 15.05.26":"ESTATUS USD A NIVEL HTS",
        "OBSERVACIONES GLOBALES YA CON R1 (ETIQUETA ANTERIOR)":"ESTATUS X USD A NIVEL GLOBAL DEL CROSS X PED",
        "ESTATUS CANTIDAD COMERCIAL A NIVEL HTS AL 15.05.26":"ESTATUS CANTIDAD COMERCIAL A NIVEL HTS"
      
    })
    [
      [
        "CLAVES IMMEX",
        "FECHA PAGO",
        "FECHA ENTRADA O PRESENTACIÓN",
        "ID AF?",
        "TIPO Y CLAVE",
        "SI EL PEDIMENTO DE LA COLUMNA 'Q' ES R1, A QUIEN RECTIFICÓ?",        
        "ESTATUS DEL PEDIMENTO A QUIEN RECTIFICO DE LA COLUMNA 'F'?",
        "SI EL PEDIMENTO DE LA COLUMNA 'Q' TUVO R1, CUÁL FUE EL NÚMERO DE R1?",        
        "ESTATUS DEL PEDIMENTO R1 DE LA COLUMNA H",
        "ESTIMACIÓN APROX DE IGI  6 FP 0",        
        "ESTIMACIÓN APROX DE IGI 6 FP 5",        
        "ESTIMACIÓN APROX DE IGI 6 FP 6",
        "SI EL PEDIMENTO DE LA COLUMNA 'Q' ES DESCARGA ESPECIFICA, A QUIÉN DESCARGO?",
        "SI EL PEDIMENTO DE LA COLUMNA 'Q' TUVO DESCARGAR ESPECIFICA, EN QUE PEDIMENTO SE DESCARGO?",
        "DS PAT-PED",
        "DS PAT-PED-HTS",        
        "DS ADU-PAT-PED",
        "DS ADU-PAT-PED-HTS",        
        "DS VALOR DOLARES",        
        "BASE VALOR DOLARES",        
        "DIFERENCIA REAL",
        "DIFERENCIA ABSOLUTA",
        "REVISIÓN DE DUPLICADOS",
        "DS CANTIDAD COMERCIAL",
        "DS UNIDAD DE MEDIDA COMERCIAL",
        "BASE CANTIDAD COMERCIAL",
        "BASE UNIDAD DE MEDIDA COMERCIAL",
        "DIFERENCIA REAL UND",
        "DIFERENCIA ABSOLUTA UND",
        "REVISIÓN DE DUPLICADOS UND",
        "ESTATUS X USD A NIVEL GLOBAL DEL CROSS X PED",
        "ESTATUS USD A NIVEL HTS",
        "ESTATUS CANTIDAD COMERCIAL A NIVEL HTS",
        "ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 1)",
        "ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 2)",
        "ESTATUS HTS A NIVEL PEDIMENTO",
        "LLAVE PED-VALORUSD"

      ]
    ]
)



In [112]:
df_ds_group.to_excel("outputs/CROSS X HTS.xlsx", index=False)
# df_ped_aux
# df_ped_aux.to_excel("outputs/CUALQUIERA.xlsx", index=False)

In [113]:
# print(repr(df_ds_group.loc[79698, "DS VALOR DOLARES"]))

In [114]:
pd.set_option("display.max_columns", None)
# df_ds_group
# df_ds_group

# display(df_ds_group.columns.tolist())

display(
    df_ds_group[
        df_ds_group["DS ADU-PAT-PED-HTS"] == "160-3687-3000255-84819005"
        
    ]
)



,CLAVES IMMEX,FECHA PAGO,FECHA ENTRADA O PRESENTACIÓN,ID AF?,TIPO Y CLAVE,"SI EL PEDIMENTO DE LA COLUMNA 'Q' ES R1, A QUIEN RECTIFICÓ?",ESTATUS DEL PEDIMENTO A QUIEN RECTIFICO DE LA COLUMNA 'F'?,"SI EL PEDIMENTO DE LA COLUMNA 'Q' TUVO R1, CUÁL FUE EL NÚMERO DE R1?",ESTATUS DEL PEDIMENTO R1 DE LA COLUMNA H,ESTIMACIÓN APROX DE IGI 6 FP 0,ESTIMACIÓN APROX DE IGI 6 FP 5,ESTIMACIÓN APROX DE IGI 6 FP 6,"SI EL PEDIMENTO DE LA COLUMNA 'Q' ES DESCARGA ESPECIFICA, A QUIÉN DESCARGO?","SI EL PEDIMENTO DE LA COLUMNA 'Q' TUVO DESCARGAR ESPECIFICA, EN QUE PEDIMENTO SE DESCARGO?",DS PAT-PED,DS PAT-PED-HTS,DS ADU-PAT-PED,DS ADU-PAT-PED-HTS,DS VALOR DOLARES,BASE VALOR DOLARES,DIFERENCIA REAL,DIFERENCIA ABSOLUTA,REVISIÓN DE DUPLICADOS,DS CANTIDAD COMERCIAL,DS UNIDAD DE MEDIDA COMERCIAL,BASE CANTIDAD COMERCIAL,BASE UNIDAD DE MEDIDA COMERCIAL,DIFERENCIA REAL UND,DIFERENCIA ABSOLUTA UND,REVISIÓN DE DUPLICADOS UND,ESTATUS X USD A NIVEL GLOBAL DEL CROSS X PED,ESTATUS USD A NIVEL HTS,ESTATUS CANTIDAD COMERCIAL A NIVEL HTS,ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 1),ESTATUS HTS A NIVEL PEDIMENTO (AUXILIAR 2),ESTATUS HTS A NIVEL PEDIMENTO,LLAVE PED-VALORUSD
79698,UNIVERSO IMMEX,6/01/2023,26/12/2022,NaN,1-IN,NO ES R1,NO SE REQUIERE,NO TUVO R1,NO SE REQUIERE,NaN,NaN,NaN,NO ES DESCARGA,NO DESCARGO,3687-3000255,3687-3000255-84819005,160-3687-3000255,160-3687-3000255-84819005,120382.0,120382.0,0.0,0.0,,14120.0,PC,14120.0,PC,0,0.0,,UNIVERSO IMMEX-100% CORRECTO EN VALOR,UNIVERSO IMMEX-100% CORRECTO EN VALOR HTS,UNIVERSO IMMEX-100% CORRECTO EN CANTIDAD HTS,FRACCION 100% CORRECTA EN DOLARES Y CANTIDAD,PEDIMENTO CON HTS 100% CORRECTOS DOLARES Y CAN...,PEDIMENTO CON HTS 100% CORRECTOS DOLARES Y CAN...,160-3687-3000255-120382
